## How to use functions from the sdg-tagger

In [ ]:
#Installs all python packages listed in the requirements file. You can skip this step if you have already done this in your envirionment.
%pip install -r requirements.txt

### Imports used in the entire file. Run this cell first!

In [ ]:
import json 
from rich import print_json #This is used for colouring when printing the output
from IPython.display import display, HTML

### Search for all targets in a specific goal, and get the results in json format
Function: `search_all_targets_in_goal`

In [ ]:
# -------- Import the function we want to use from src --------
from src import search_in_text_for_one_sdg 

# ------- Define the SDG number and input text --------
SDG_NR = 2
TEXT = "This is a title about extreme poverty and ending extreme hunger in least developed countries."
#include_indexed_results = False # Set this to True if you want to also see the index (position) in the text where the search terms were found

# ------------------ Searching ----------------------
results = search_in_text_for_one_sdg(sdg_nr=SDG_NR, input_text=TEXT)

# --------------- Printing the results ---------------
print_json(json.dumps(results, indent=4)) # With coloured output (works best with light colour themes in your editor)
#print(json.dumps(results, indent=4)) # Use this to have the normal uncoloured output


### Search all goals on one text
Function: `search_all_goals`

In [ ]:
from src import search_all_goals

text = "This is a title about extreme poverty and ending extreme hunger in least developed countries."
results = search_all_goals(text)
print(json.dumps(results, indent=4))
#print_json(json.dumps(results, indent=4)) # Use this if you want the coloured output

### Trigger all country category searches
Function: `all_country_searches`

In [ ]:
from src import run_all_country_searches

text = 'This should be true for LDC, LMIC and LDS because I mention Burkina Faso.'
results_boolean = run_all_country_searches(input_text=text)
print(json.dumps(results_boolean, indent=4))
#print_json(json.dumps(results_boolean, indent=4)) # Use this if you want the coloured output

### Run search on one specific target on an entire dataframe

In [ ]:
# Enter you file path, the SDG and target you want to search for, and the columns in which you want to search
FILE_NAME = r"C:\Users\xxx\TESTdata_titles-abstracts-SDGlabels_2020-2023.csv"
SDG_NR = 14 # Must be a number
TARGET = '1' # Must be a string corresponding to a target in the SDG
COLUMNS = ['result_title', 'abstract'] # Text columns to search
VIEW = ["sdg"] #Use "sdg" to view the target results, and add "extra" like so ["sdg", "extra"] to view the country and pre-search results
EXTRA_COLUMNS = ["SDG_target_topic"] # Extra columns for viewing
INCLUDE_FALSE_VALUES = False # Whether to show or hide result rows where the target is not found
DO_NOT_RUN_COUNTRIES_SEARCH = True #Set this to True only if none of the phrases in the target refers to any of the country searches! This will force the country searches not to run to make the search way faster. 

In [ ]:
from src import dataframe_search_target
from src.consts import COUNTRIES
import pandas as pd

# Read the data from your file
data_raw = pd.read_csv(FILE_NAME, sep=",")
data_raw = data_raw[:1000]
data = data_raw[COLUMNS]

# Perform the search
result_df = data_raw[COLUMNS]
for index in range(len(COLUMNS)):
    text_col = COLUMNS[index]
    print(f'Searching in {text_col}')
    if DO_NOT_RUN_COUNTRIES_SEARCH:
        countries={'LDC': False, 'SIDS': False, 'LDS': False, 'LMIC': False}
    else:
        countries = False
    col_result = dataframe_search_target(data[[text_col]], sdg_nr=SDG_NR, target=TARGET, text_column=text_col, countries=countries)
    col_result = col_result.drop(text_col, axis=1)

    if index==0:
        original_columns = [col for col in col_result.columns if col not in COLUMNS]
        
    col_result = col_result.add_suffix(f'_{text_col}')
    result_df = pd.merge(result_df, col_result, left_index=True, right_index=True)
    prev_col = text_col

# Create a new, interleaved column order
interleaved_cols = []
for col in original_columns:
    for text_col in COLUMNS:
        interleaved_cols.append(col + f'_{text_col}')

#final_cols = COLUMNS + EXTRA_COLUMNS + interleaved_cols
final_df = pd.concat([data_raw[EXTRA_COLUMNS + COLUMNS], result_df[interleaved_cols]], axis=1)

#Define SDG cols
sdg_cols = [
    col for col in interleaved_cols
    if col.startswith(f"tempsdg{f'{int(SDG_NR):02d}'}")
]

#Filter rows depending only on if they have True in SDG result (not in the pre-search)
if not INCLUDE_FALSE_VALUES:
    if len(sdg_cols) > 0:
        mask = (final_df[sdg_cols].fillna(False) == False).all(axis=1)
        final_df = final_df[~mask]

#Choose which columns to view
view = set(VIEW) if len(VIEW) > 0 else {"sdg", "extra"}

sdg_enabled = "sdg" in view
extra_enabled = "extra" in view

cols_to_view = []

if sdg_enabled:
    cols_to_view.extend(sdg_cols)

extra_cols = [col for col in interleaved_cols if col not in sdg_cols]

if extra_enabled:
    cols_to_view.extend(extra_cols)

display_df = final_df[EXTRA_COLUMNS + COLUMNS + cols_to_view]
print(display_df.columns)

print(f"Total rows in dataset: {len(data_raw)}")
print(f"Total results: {len(display_df)}")

# Colours all values that are not nan or False
# Only style the SDG columns for coloring, and text columns for width
text_cols_to_style = [col for col in COLUMNS if col in final_df.columns]

def color_true_green(val):
    if (pd.notna(val)) and (val is not False):
        return f'background-color: green'
    
styled_df = display_df.style.map(color_true_green, subset=cols_to_view).set_properties(
        subset=text_cols_to_style,   # your long-text column
        **{
            "min-width": "300px",
            "max-width": "300px",
            "white-space": "normal"
        })
styled_df

display(HTML(f"""
<div style="max-height: 700px; overflow-y: auto;">
    {styled_df.to_html()}
</div>
"""))


### Run search for all sdgs on an entire dataframe
Function: `dataframe_search`

Box 1 contains search parameters, and what you want to view in the final table. You can alter everything here.

Box 2 contains the search itself - here the only thing you should alter are lines that determine how you subset your data (i.e. run the search on a part of it). You can do this by subsetting a number of rows (Option 1), or number of rows as well as original SDG result (Option 2):

- To use all your data, uncomment out all code in Option 1, comment out Option 2, and alter the line "Choose which rows to search".
- If wanting to search a subset of your data by previous SDG result, or only looking for Norwegian results, alter in Option 2 the lines:
  - "data_raw_sdg = ..." - filter by an SDG
  - "data_raw_sdgtarget = ..." - filter by a target
  - "data_raw_sdg_norsk = ..." - filter by an SDG and only norwegian results
  - "data_raw_sdgtarget_norsk = ..." - filter by an SDG target and only norwegian results
  - THEN remember to change it to the one you want to use, in this line! `data_raw = data_raw_sdgtarget[:10]` (and how many results)

In [ ]:
# Enter you file path, the SDGs you want to search for, and the columns in which you want to search
FILE_NAME = r"C:\Users\xxx\TESTdata_titles-abstracts-SDGlabels_2020-2023.csv"
SDGS = [1,2] #Choose which SDGs to search for
COLUMNS = ['result_title', 'abstract'] # Choose columns of your data to be searched
VIEW = ["sdg01_01", "sdg01_02", "sdg01_03", "sdg02_01"] #Choose if you want to only examine specific columns when the search is finished - if empty, all will be shown. Can include LMIC etc.
EXTRA_COLUMNS = ["result_id", "SDG_target_topic"] #Choose additional columns from your data to be included in the final table.
INCLUDE_FALSE_VALUES = True #Do you want results that are not found by the search to be included?

In [ ]:
from src import dataframe_search
from src.consts import COUNTRIES
import pandas as pd

# Option 1: Use all data: Read the data from your file and *** choose which rows to search ***
# If you wish to use this, comment out Option 2, and uncomment this
#data_raw = pd.read_csv(FILE_NAME, sep=",")
#data_raw = data_raw[:10] #Choose which rows to search
#data = data_raw[COLUMNS]

# Option 2: Use a subset of data: Read the data from your file and *** filter by earlier SDG result, and choose which rows to search ***
data_raw = pd.read_csv(FILE_NAME, sep=",")

data_raw_sdg = data_raw[data_raw['SDG_topic'].str.contains("SDG01", na=False)] #Choose SDG - take a subset of your data by previous SDG result. 
data_raw_sdgtarget = data_raw[data_raw['SDG_target_topic'].str.contains("SDG01_01", na=False)] #Choose SDG - take a subset of your data by previous SDG target result. 
    # For multiple, combine like so: "SDG01|SDG02|...". The names of the targets are 01 onwards, then "_0a", "_b", and "_c".
data_raw_sdg_norsk = data_raw[
    data_raw['SDG_topic'].str.contains("SDG15", na=False) &     #Choose SDG - take a subset of your data by previous SDG result. 
    data_raw['language'].str.contains("Norwegian", na=False)    #Choose language (currently = norwegian)
] 
data_raw_sdgtarget_norsk = data_raw[
    data_raw['SDG_target_topic'].str.contains("SDG15_c", na=False) &   #Choose SDG - take a subset of your data by previous SDG target result.
    data_raw['language'].str.contains("Norwegian", na=False)            #Choose language (currently = norwegian)
]

data_raw = data_raw_sdgtarget[:10] #Choose which/how many rows to search, and which dataset (data_raw_sdg or data_raw_sdgtarget) to use! 
data = data_raw[COLUMNS]

# Perform the search
result_df = data_raw[COLUMNS]
for index in range(len(COLUMNS)):
    text_col = COLUMNS[index]
    print(f'Searching in {text_col}')
    col_result = dataframe_search(data[[text_col]], sdg_list=SDGS, text_column=text_col)
    col_result = col_result.drop(text_col, axis=1)

    if index==0:
        original_columns = [col for col in col_result.columns if col not in COLUMNS]
        
    col_result = col_result.add_suffix(f'_{text_col}')
    result_df = pd.merge(result_df, col_result, left_index=True, right_index=True)
    prev_col = text_col

# Create a new, interleaved column order
interleaved_cols = []
for col in original_columns:
    for text_col in COLUMNS:
        interleaved_cols.append(col + f'_{text_col}')

if len(VIEW) > 0:
    cols_to_view = [col for col in interleaved_cols if any(v in col for v in VIEW)]
    final_cols = COLUMNS + EXTRA_COLUMNS + cols_to_view
else:
    final_cols = COLUMNS + EXTRA_COLUMNS + interleaved_cols
final_df = pd.concat([data_raw[EXTRA_COLUMNS + COLUMNS], result_df[cols_to_view]], axis=1)

if not INCLUDE_FALSE_VALUES:
    # Creates a mask to filter only on rows that contains some true values
    filter_cols = [x for x in cols_to_view if not any(y in x for y in [z['name'] for z in COUNTRIES])]
    falsy_mask = (final_df[filter_cols].fillna(False) == False).all(axis=1)
    final_df = final_df[~falsy_mask]

# Colours all values that are not nan or False
# Only style the SDG columns for coloring, and text columns for width
text_cols_to_style = [col for col in COLUMNS if col in final_df.columns]

def color_true_green(val):
    if (pd.notna(val)) and (val is not False):
        return f'background-color: green'
    
styled_df = final_df.style.map(color_true_green, subset=cols_to_view).set_properties(
        subset=text_cols_to_style,   # your long-text column
        **{
            "min-width": "300px",
            "max-width": "300px",
            "white-space": "normal"
        })
styled_df

display(HTML(f"""
<div style="max-height: 700px; overflow-y: auto;">
    {styled_df.to_html()}
</div>
"""))

Saving the styled results to excel

In [ ]:
file_path = 'styled_output.xlsx'
try:
    styled_df.to_excel(file_path, engine='openpyxl', index=False)
    print(f"Successfully saved styled DataFrame to {file_path}")
except ImportError as e:
    print(f"Error: {e}. Please ensure 'openpyxl' or 'xlsxwriter' is installed.")
except Exception as e:
    print(f"An error occurred: {e}")

### Getting help to use a function correct:
Use `help(<function name>)` to get info about the function you are using

In [ ]:
from src import run_all_country_searches

help(run_all_country_searches)